# Reacher Atlas-DeePC: 500-Seed Four-Method Evaluation

**500 evaluation seeds: 10001--10500. Four methods, 2,000 rollouts.**

Ranks were selected using **development seeds 211--310** at 320 steps. This
evaluation freezes those ranks and uses **T_sim = 300 controlled steps**.
The chart counts below were chosen by the user. Source development studies:
[Combined chart-count and rank ablation](../ablations/ablation_chart_count_and_rank.ipynb).

| Output dimension | Graph guidance | Frozen rank | Charts |
|---|---|---:|---:|
| 8 | No | 53 | 64 |
| 8 | Yes | 57 | 96 |
| 10 | No | 64 | 80 |
| 10 | Yes | 62 | 64 |

Each case uses its own frozen development atlas. No reclustering or rank search
is performed here. Controller weights and reference policies remain unchanged.
These 500 seeds were already evaluated with K=48: they are disjoint from
development, but this is a **repeated evaluation**, not a new untouched test set.
Do not retune using these evaluation results. Since both chart counts and ranks
differ, the four-method comparison is not a controlled graph-only ablation.

## Protocol

The objective contains only tracking error, input effort, and initial-output
slack, with $\lambda_{\mathrm{ini}}=10^5$. There are no coefficient or input-change
penalties, coefficient boxes, affine charts, or extra terminal penalties.
Clarabel uses the existing solver settings. All coordinates are unstandardized.
Charts are selected by projection residual using the most recent **L=12**
complete input/output samples, with no coordinate truncation.

Unguided methods use a fixed nearest eligible offline terminal configuration.
Guided methods retain chordal graph costs divided by square-root transition
counts, graph waypoints, and iterative **N=10** terminal-candidate selection.
Thus the guidance comparison includes the existing terminal-reference policy
difference; it is not an isolated waypoint-only ablation.

Each seed uses the same Reacher reset across methods, 12 zero-input warmup
steps, then 300 controlled steps. Default resets cover the target disk but only
small initial joint-angle perturbations near zero; this is not a test over
arbitrary arm configurations. Evaluation seeds are disjoint from development
and offline episode seeds.

Strict success requires mean fingertip error below 10 mm and maximum error
below 20 mm in **both** the final 30 and final 100 steps. Error quantiles are
across rollout final-100 mean errors. Median rollout RMSE is the median of
500 individual RMSEs, each using all 300 controlled steps relative to the true
target, excluding warmup. Failed solves remain in the statistics.

## Run or Resume

Select the **gym-robotics-mujoco** kernel. Set `RUN_EXPERIMENTS = True` and Run All.
Runs are **serial with one numerical thread**, for comparable timings. Wait for
other heavy experiments to finish first. Interrupt
the run cell to stop; complete seed files are reused on the next run.
With the flag False, the notebook loads the saved protocol and displays progress
(or prepares a protocol if none exists). No test rollouts or videos run by default.

The shared controller now exposes only tracking, input effort and initial-slack
costs. Existing results and their frozen source signatures are retained unchanged.
Saved results can be viewed without revalidating them against edited source;
a new run still enforces source compatibility and may require a fresh result root.


In [ ]:
from pathlib import Path
import os, sys
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[name] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/atlas_deepc_mpl")
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/atlas_deepc").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import importlib
from atlas_deepc.benchmarks import reacher_heldout as shared_runner
from atlas_deepc.benchmarks import reacher_mixed_k as study
from atlas_deepc.reporting import current_reacher_baseline as visuals

# Reload dependencies first: an existing kernel may still hold the K=48 runner.
importlib.reload(shared_runner)
importlib.reload(study)
importlib.reload(visuals)

RUN_EXPERIMENTS = False
PLOT_CHART_CLOUDS = False
RENDER_MEDIA = False
RESULT_ROOT = study.RESULT_ROOT
REPORT_ROOT = RESULT_ROOT / "baseline_report"
T_SIM = study.CONTROL_STEPS
assert T_SIM == 300
saved_manifest = RESULT_ROOT / "manifest.json"
protocol = (json.loads(saved_manifest.read_text())
            if saved_manifest.exists() and not RUN_EXPERIMENTS else study.prepare())
print(f"T_sim={T_SIM}; seeds {study.SEEDS[0]}--{study.SEEDS[-1]}; results: {RESULT_ROOT}")
display(pd.DataFrame(study.report()["progress"]))


T_sim=300; seeds 10001--10500; results: /Users/kumiko/Projects/atlas_deepc_reacher/experiments/reacher/four_baselines_500_mixed_k_t300_v1


,outputs,graph,rank,charts,control_steps,completed,requested
0,8,False,53,64,300,0,500
1,8,True,57,96,300,0,500
2,10,False,64,80,300,0,500
3,10,True,62,64,300,0,500


In [ ]:
if RUN_EXPERIMENTS:
    import signal, subprocess
    child_env = dict(os.environ, PYTHONPATH=str(ROOT / "src"))
    process = subprocess.Popen(
        [sys.executable, "-u", "-m", "atlas_deepc.benchmarks.reacher_mixed_k"],
        cwd=ROOT, env=child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError("Evaluation stopped. See the error above; saved seeds are retained.")
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
        print("Stopped; completed seeds remain saved. Run this cell again to resume.")
    finally:
        process.stdout.close()
else:
    print("No rollouts launched. Set RUN_EXPERIMENTS = True to run/resume.")


p=8, terminal: 1/500, seed=10001, success=True, final100=2.746 mm
p=8, terminal: 2/500, seed=10002, success=True, final100=1.914 mm
p=8, terminal: 3/500, seed=10003, success=True, final100=5.701 mm
p=8, terminal: 4/500, seed=10004, success=True, final100=0.905 mm
p=8, terminal: 5/500, seed=10005, success=True, final100=2.810 mm
p=8, terminal: 6/500, seed=10006, success=True, final100=0.833 mm
p=8, terminal: 7/500, seed=10007, success=True, final100=9.997 mm
p=8, terminal: 8/500, seed=10008, success=True, final100=5.114 mm
p=8, terminal: 9/500, seed=10009, success=True, final100=2.253 mm
p=8, terminal: 10/500, seed=10010, success=True, final100=1.126 mm
p=8, terminal: 11/500, seed=10011, success=True, final100=1.837 mm
p=8, terminal: 12/500, seed=10012, success=True, final100=2.226 mm
p=8, terminal: 13/500, seed=10013, success=False, final100=30.623 mm
p=8, terminal: 14/500, seed=10014, success=True, final100=8.948 mm
p=8, terminal: 15/500, seed=10015, success=True, final100=1.475 mm
p=

## Evaluation Results

The final table is generated only after all four 500-seed panels are complete.
Partial progress is labeled below. Success rates retain 0.2-percentage-point
resolution. The LaTeX fragment requires `booktabs` in the paper preamble.


In [ ]:
report = study.report()
display(pd.DataFrame(report["progress"]))
if report["complete"]:
    columns = report["columns"]
    labels = [f"p={c['output_dimension']}, " + ("graph" if c["graph_guidance"] else "no graph") for c in columns]
    fields = ["charts", "rank", "control_steps", "success_rate", "median_error_mm", "p90_error_mm", "p95_error_mm", "p99_error_mm",
              "median_rollout_rmse_mm", "median_time_ms", "p95_time_ms", "p99_time_ms"]
    display(pd.DataFrame({label: {k: c[k] for k in fields} for label, c in zip(labels, columns)}))
    print((RESULT_ROOT / "table_heldout500.tex").read_text())
else:
    display(Markdown("**Incomplete evaluation. No final test-performance table yet.**"))
for c in report["columns"]:
    print(f"p={c['output_dimension']}, {c['reference_mode']}: {len(c['failed_seeds'])} failed seeds")
    print(c["failed_seeds"])


,outputs,graph,rank,charts,control_steps,completed,requested
0,8,False,53,64,300,500,500
1,8,True,57,96,300,500,500
2,10,False,64,80,300,500,500
3,10,True,62,64,300,500,500


,"p=8, no graph","p=8, graph","p=10, no graph","p=10, graph"
charts,64.000000,96.000000,80.000000,64.000000
rank,53.000000,57.000000,64.000000,62.000000
control_steps,300.000000,300.000000,300.000000,300.000000
success_rate,90.000000,94.200000,83.800000,98.800000
median_error_mm,1.809298,1.855341,1.531974,1.910387
p90_error_mm,10.047504,4.608627,232.290640,3.813674
p95_error_mm,218.267459,13.159995,296.355326,4.766760
p99_error_mm,299.295277,48.231939,371.368400,10.550875
median_rollout_rmse_mm,55.806776,76.655420,57.180483,62.636651
median_time_ms,20.012563,19.917084,23.028605,22.652084


\begin{table}[htbp]
    \centering
    \small
    \begin{tabular}{lrrrr}
        \toprule
        Output dimension & \multicolumn{2}{c}{$p=8$} & \multicolumn{2}{c}{$p=10$} \\
        \cmidrule(lr){2-3}\cmidrule(lr){4-5}
        Graph guidance & No & Yes & No & Yes \\
        \midrule
        Charts $K$ & 64 & 96 & 80 & 64 \\
        Selected rank & 53 & 57 & 64 & 62 \\
        Success rate $\uparrow$ & 90.0\% & 94.2\% & 83.8\% & \textbf{98.8\%} \\
        Median error (mm) $\downarrow$ & 1.81 & 1.86 & \textbf{1.53} & 1.91 \\
        90th-percentile error (mm) $\downarrow$ & 10.048 & 4.609 & 232.291 & \textbf{3.814} \\
        95th-percentile error (mm) $\downarrow$ & 218.267 & 13.160 & 296.355 & \textbf{4.767} \\
        99th-percentile error (mm) $\downarrow$ & 299.295 & 48.232 & 371.368 & \textbf{10.551} \\
        Median rollout RMSE (mm) $\downarrow$ & \textbf{55.81} & 76.66 & 57.18 & 62.64 \\
        \midrule
        \multicolumn{5}{l}{\textit{Online computation time (ms/step)}} \

## Complete-Trajectory Physical Chart Clouds

Optional offline visualization of the frozen atlas. Blue shows the two past
poses and orange the ten future poses of each sampled trajectory. The dark-blue
representative is for visualization only; online selection uses projection
residuals. The same color palette is used for every chart.


In [ ]:
if PLOT_CHART_CLOUDS:
    plt.rcParams.update({"font.size": 11, "mathtext.fontset": "cm"})
    for p, mode, rank in study.CASES:
        figure = visuals.chart_clouds(p, RESULT_ROOT, REPORT_ROOT / f"output{p}/{mode}",
                                     atlas_file=study.atlas_path(RESULT_ROOT, p, mode))
        plt.show()
        plt.close(figure)


## Success and Failure Replays

Optional: after evaluation, render the lowest-numbered success and failure seed
for each method (where available). These are post-evaluation illustrations,
not additional selection or tuning. Simulator replays use physical 1x speed;
diagnostic dashboards use 0.25x. Saved actions are replayed with 12 zero-input
warmup steps. No optimization is rerun to create the videos.


In [ ]:
if RENDER_MEDIA and report["complete"]:
    from atlas_deepc.reporting import render_saved_rollout as replay
    from atlas_deepc.reporting import render_routed_deepc_dashboard as dashboard
    entries = []
    for c in report["columns"]:
        p, mode = c["output_dimension"], c["reference_mode"]
        failed = set(c["failed_seeds"])
        picked = ([next(s for s in study.SEEDS if s not in failed)] if len(failed) < 500 else [])
        picked += [min(failed)] if failed else []
        directory = RESULT_ROOT / f"output{p}/{mode}"
        panel = json.loads((directory / "summary.json").read_text())
        rows = {r["seed"]: r for r in panel["results"]}
        for seed in picked:
            source = directory / f"seed{seed}_control_results.npz"
            media_dir = REPORT_ROOT / f"videos/output{p}/{mode}"
            media_dir.mkdir(parents=True, exist_ok=True)
            simulator = media_dir / f"seed{seed}_simulator.mp4"
            movie = media_dir / f"seed{seed}_dashboard_025x.mp4"
            replay.render_saved_rollout(source, seed, simulator, fps=50, frame_stride=1,
                warmup_zeros=12, verification_path=simulator.with_suffix(".json"),
                output_from_obs=lambda obs, p=p: replay.pipe.reacher_output_from_obs(obs)[:p])
            dashboard.render_dashboard(source, study.atlas_path(RESULT_ROOT, p, mode), movie,
                fps=25, frame_stride=2, play_speed=.25, end_step=T_SIM,
                evaluation_label=f"500-seed evaluation | p={p} | K={c['charts']} | rank={c['rank']}")
            entries.append({"output_dimension": p, "mode": mode, "seed": seed,
                "success": seed not in failed, "final100_mm": 1000*rows[seed]["mean_last_100"],
                "simulator": str(simulator), "dashboard": str(movie)})
    visuals.gallery(entries, "simulator")
    visuals.gallery(entries, "dashboard")
elif RENDER_MEDIA:
    print("Finish the evaluation before selecting illustrative replays.")


## Saved Artifacts

- `manifest.json`: frozen seeds, ranks, configurations, data/atlas/source hashes.
- `frozen_source/`: controller and evaluation source snapshot.
- `output8/` and `output10/`: per-seed traces, graph/projection geometry, summaries.
- `evaluation_report.json`: completion state, metrics, failed seeds and rollout RMSEs.
- `table_heldout500.tex`: final four-column paper table, written only on completion.

Results are stored separately in `four_baselines_500_mixed_k_t300_v1`.
Development results and offline data remain intact. The previous K=48
raw 500-seed rollout data and generated videos were removed to free storage;
summary tables and provenance remain. Old raw-trace analyses require rerunning.


## Matched-Atlas Guidance: Seven Figure Seeds

Run **only seven guided rollouts**, with **p=8, K=64, rank=53, T_sim=300**:
**10206, 10349, 10144, 10445, 10330, 10255, 10449**. Seed 10449 is the
matched-atlas success-to-failure regression example. Reuse the exact frozen atlas
and controller weights of the unguided baseline; do not recluster or retune.
The existing unguided traces provide the paired comparison and are not rerun.
All other settings remain unchanged: full-window projection selection, Clarabel,
12 zero-input warmup steps, and chordal graph costs.

This matches K and rank, but tests the **guidance package**, including iterative
N=10 terminal selection, not waypoints alone. These figure-selected seeds are
illustrative and must not be used as an unbiased success-rate estimate.

Run the setup cell above, then set `RUN_MATCHED_GUIDANCE = True` below and run
this section only. Wait until the Select-DPC timing run finishes. Full trajectories
are saved separately in `experiments/reacher/figure_guidance_p8_k64_r53_t300_v1`.
Completed seeds are reused after interruption; the 500-seed results are untouched.


In [ ]:
from atlas_deepc.ablations import figure_guidance_matched as matched_guidance
importlib.reload(matched_guidance)
RUN_MATCHED_GUIDANCE = False
print("Figure seeds:", matched_guidance.SEEDS)
print("Guided traces:", matched_guidance.RESULT_ROOT / "output8/graph")
if RUN_MATCHED_GUIDANCE:
    import signal, subprocess
    matched_process = subprocess.Popen(
        [sys.executable, "-u", "-m", "atlas_deepc.ablations.figure_guidance_matched"],
        cwd=ROOT, env=dict(os.environ, PYTHONPATH=str(ROOT / "src")),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True)
    try:
        for line in matched_process.stdout:
            print(line, end="", flush=True)
        if matched_process.wait():
            raise RuntimeError("Matched guidance stopped; see output above. Completed seeds are retained.")
    except KeyboardInterrupt:
        try:
            os.killpg(matched_process.pid, signal.SIGINT)
        except ProcessLookupError:
            pass
        try:
            matched_process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(matched_process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            matched_process.wait()
        print("Stopped; rerun this cell to resume the seven-seed panel.")
    finally:
        matched_process.stdout.close()
else:
    print("No new rollouts launched. Enable RUN_MATCHED_GUIDANCE when ready.")


Figure seeds: (10206, 10349, 10144, 10445, 10330, 10255, 10449)
Guided traces: /Users/kumiko/Projects/atlas_deepc_reacher/experiments/reacher/figure_guidance_p8_k64_r53_t300_v1/output8/graph


In [ ]:
matched_rows = matched_guidance.paired_rows()
display(pd.DataFrame(matched_rows).reindex(columns=["seed", "unguided_success", "guided_success",
                   "unguided_final100_mm", "guided_final100_mm"]))
print("Saved paired results:", matched_guidance.RESULT_ROOT / "paired_results.json")


seed,unguided_success,guided_success,unguided_final100_mm,guided_final100_mm
10206,False,True,377.344119,1.845311
10349,False,True,375.499437,0.424394
10144,False,False,216.761017,16.941154
10445,False,True,191.072191,1.424071
10330,True,True,0.818220,0.818218
10255,True,True,1.865330,3.038914
10449,True,False,2.258558,60.807853


Saved paired results: /Users/kumiko/Projects/atlas_deepc_reacher/experiments/reacher/figure_guidance_p8_k64_r53_t300_v1/paired_results.json
